# <span style="color: var(--vscode-foreground); background-color: yellow;"><u style=""><b>CHAPTER 4 &amp; 5 Proposition</b></u></span>

**<mark>\### Proposition 1: Find the Employee Who Processed the Most Orders #### Problem: Retrieve the employee who has processed the highest number of orders in the database. This helps identify the top-performing salesperson.</mark>**

In [23]:
USE WideWorldImporters;
WITH TopSalesperson AS (
    SELECT TOP 1 SalespersonPersonID
    FROM Sales.Orders
    WHERE SalespersonPersonID IS NOT NULL  
    GROUP BY SalespersonPersonID
    ORDER BY COUNT(OrderID) DESC
)
SELECT PersonID, FullName
FROM Application.People
WHERE IsSalesperson = 1  
AND PersonID = (SELECT SalespersonPersonID FROM TopSalesperson);


(1 row affected)

Total execution time: 00:00:00.048

PersonID,FullName
16,Archer Lamble


**<mark>\### Proposition 2: Find Unique Countries with Orders</mark>**  

**<mark>\#### Problem:</mark>**  

**<mark>Identify all unique countries that have placed at least one order.</mark>**  

**<mark>This helps analyze geographical distribution of customers.</mark>**

In [13]:
USE WideWorldImporters;

WITH OrderCountries AS (
    SELECT DISTINCT Country.CountryName
    FROM Sales.Orders O
    JOIN Sales.Customers C ON O.CustomerID = C.CustomerID
    JOIN Application.Cities City ON C.DeliveryCityID = City.CityID
    JOIN Application.StateProvinces SP ON City.StateProvinceID = SP.StateProvinceID
    JOIN Application.Countries Country ON SP.CountryID = Country.CountryID
)
SELECT * FROM OrderCountries;


(1 row affected)

Total execution time: 00:00:00.196

CountryName
United States


**<mark>\### Proposition 3: Find Customers Who Ordered Again Within 3 Days #### Problem: Detect customers who placed another order within three days of their previous order. This helps in customer behavior analysis for targeted marketing.</mark>**

In [14]:
USE WideWorldImporters;
WITH RepeatOrders AS (
    SELECT O1.OrderID, O1.CustomerID, O1.OrderDate
    FROM Sales.Orders O1
    WHERE EXISTS (
        SELECT 1
        FROM Sales.Orders O2
        WHERE O2.CustomerID = O1.CustomerID
        AND O2.OrderDate BETWEEN O1.OrderDate AND DATEADD(DAY, 3, O1.OrderDate)
        AND O2.OrderID <> O1.OrderID
    )
)
SELECT * FROM RepeatOrders;


(32065 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:01.023

OrderID,CustomerID,OrderDate
895,169,2013-01-16
2685,169,2013-02-26
72839,169,2016-05-20
72931,169,2016-05-20
63633,169,2015-12-28
64115,169,2016-01-04
64162,169,2016-01-04
62016,169,2015-12-01
62256,169,2015-12-03
62277,169,2015-12-03


**<mark>\### Proposition 4: Compute Running Total of Order Amounts #### Problem: Track cumulative sales per customer by computing a running total of their order amounts. This helps businesses monitor spending patterns over time.</mark>**

In [16]:
USE WideWorldImporters;
WITH RunningOrderTotals AS (
    SELECT O1.CustomerID, O1.OrderID, O1.OrderDate,
        (SELECT SUM(OL.Quantity * OL.UnitPrice)
        FROM Sales.OrderLines OL
        WHERE OL.OrderID = O2.OrderID) AS RunningTotal
    FROM Sales.Orders O1
    LEFT JOIN Sales.Orders O2 ON O1.CustomerID = O2.CustomerID AND O2.OrderDate <= O1.OrderDate
)
SELECT * FROM RunningOrderTotals
ORDER BY CustomerID, OrderDate;


(4274538 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:19.881

CustomerID,OrderID,OrderDate,RunningTotal
1,2934,2013-03-04,690.00
1,3482,2013-03-12,3066.00
1,3482,2013-03-12,690.00
1,3651,2013-03-14,253.00
1,3651,2013-03-14,3066.00
1,3651,2013-03-14,690.00
1,4064,2013-03-21,690.00
1,4064,2013-03-21,3066.00
1,4064,2013-03-21,253.00
1,4064,2013-03-21,2614.75


**<mark>\### Proposition 5: Identify Missing Order Numbers #### Problem: Find missing order numbers in the sequence to check for data integrity issues.</mark>**

In [17]:
USE WideWorldImporters;
WITH ExpectedNumbers AS (
    SELECT ROW_NUMBER() OVER (ORDER BY OrderID) AS ExpectedID, OrderID
    FROM Sales.Orders
)
SELECT ExpectedID, OrderID
FROM ExpectedNumbers
WHERE ExpectedID <> OrderID;


(0 rows affected)

Total execution time: 00:00:00.058

ExpectedID,OrderID


**<mark>\### Proposition 6: Rank the Most Populated Cities per Country</mark>**  

**<mark>\#### Problem:</mark>**  

**<mark>Identify the top three most populated cities in each country.</mark>**

In [18]:
USE WideWorldImporters;

WITH RankedCities AS (
    SELECT 
        C.CityID, 
        C.CityName, 
        SP.CountryID, 
        C.LatestRecordedPopulation,
        RANK() OVER (PARTITION BY SP.CountryID ORDER BY C.LatestRecordedPopulation DESC) AS RankInCountry
    FROM Application.Cities C
    JOIN Application.StateProvinces SP ON C.StateProvinceID = SP.StateProvinceID
)
SELECT * 
FROM RankedCities
WHERE RankInCountry <= 3;


(3 rows affected)

Total execution time: 00:00:00.246

CityID,CityName,CountryID,LatestRecordedPopulation,RankInCountry
24161,New York,230,8175133,1
20005,Los Angeles,230,3792621,2
6330,Chicago,230,2695598,3


<mark>**\### Proposition 7: Compute Running Total of Supplier Transactions**</mark>  

<mark>**\#### Problem:**</mark>  

<mark>**Track cumulative transaction totals per supplier.**</mark>

In [19]:
USE WideWorldImporters;

WITH SupplierTransactions AS (
    SELECT 
        ST.SupplierID, 
        S.SupplierName, 
        ST.TransactionDate, 
        ST.TransactionAmount,
        SUM(ST.TransactionAmount) OVER (
            PARTITION BY ST.SupplierID 
            ORDER BY ST.TransactionDate ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS RunningTotal
    FROM Purchasing.SupplierTransactions ST
    JOIN Purchasing.Suppliers S ON ST.SupplierID = S.SupplierID
)
SELECT * FROM SupplierTransactions;


(2438 rows affected)

Total execution time: 00:00:00.131

SupplierID,SupplierName,TransactionDate,TransactionAmount,RunningTotal
1,A Datum Corporation,2016-01-04,3762.00,3762.00
1,A Datum Corporation,2016-01-04,-5956.50,-2194.50
1,A Datum Corporation,2016-01-04,2194.50,0.00
1,A Datum Corporation,2016-01-05,7524.00,7524.00
1,A Datum Corporation,2016-01-06,4639.80,12163.80
1,A Datum Corporation,2016-01-07,9405.00,21568.80
1,A Datum Corporation,2016-01-11,-21568.80,0.00
2,"Contoso, Ltd.",2013-01-02,360.53,360.53
2,"Contoso, Ltd.",2013-01-07,-360.53,0.00
4,"Fabrikam, Inc.",2013-01-02,24991.80,24991.80


**<mark>\### Proposition 8: Compare Customer Growth Year-over-Year</mark>**  

**<mark>\#### Problem:</mark>**  

**<mark>Analyze yearly customer count changes by calculating the difference from the previous year.</mark>**

In [20]:
USE WideWorldImporters;
WITH YearlyCustomerCounts AS (
    SELECT 
        YEAR(OrderDate) AS OrderYear, 
        COUNT(DISTINCT CustomerID) AS NumCustomers
    FROM Sales.Orders
    GROUP BY YEAR(OrderDate)
)
SELECT 
    OrderYear, 
    NumCustomers, 
    LAG(NumCustomers) OVER (ORDER BY OrderYear) AS PrevYearCustomers,
    NumCustomers - LAG(NumCustomers) OVER (ORDER BY OrderYear) AS Growth
FROM YearlyCustomerCounts;


(4 rows affected)

Total execution time: 00:00:00.121

OrderYear,NumCustomers,PrevYearCustomers,Growth
2013,625,NULL,NULL
2014,640,625,15
2015,657,640,17
2016,663,657,6


**<mark>\### Proposition 9: Get First and Last Recorded Temperature per Vehicle \#### Problem: Retrieve the first and last recorded temperature per vehicle to track temperature trends.</mark>**

In [21]:
USE WideWorldImporters;
WITH VehicleTemperatureHistory AS (
    SELECT 
        VehicleRegistration, 
        RecordedWhen, 
        Temperature,
        FIRST_VALUE(Temperature) OVER (
            PARTITION BY VehicleRegistration ORDER BY RecordedWhen
        ) AS FirstRecordedTemp,
        LAST_VALUE(Temperature) OVER (
            PARTITION BY VehicleRegistration ORDER BY RecordedWhen 
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS LastRecordedTemp
    FROM Warehouse.VehicleTemperatures
)
SELECT * FROM VehicleTemperatureHistory;


(65998 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:01.477

VehicleRegistration,RecordedWhen,Temperature,FirstRecordedTemp,LastRecordedTemp
WWI-321-A,2016-01-01 07:00:00.0000000,3.58,3.58,4.08
WWI-321-A,2016-01-01 07:00:00.0000000,3.40,3.58,4.08
WWI-321-A,2016-01-01 07:00:34.0000000,4.85,3.58,4.08
WWI-321-A,2016-01-01 07:00:34.0000000,4.98,3.58,4.08
WWI-321-A,2016-01-01 07:02:20.0000000,3.78,3.58,4.08
WWI-321-A,2016-01-01 07:02:20.0000000,4.03,3.58,4.08
WWI-321-A,2016-01-01 07:02:47.0000000,4.15,3.58,4.08
WWI-321-A,2016-01-01 07:02:47.0000000,3.55,3.58,4.08
WWI-321-A,2016-01-01 07:05:39.0000000,4.51,3.58,4.08
WWI-321-A,2016-01-01 07:05:39.0000000,3.02,3.58,4.08


**<mark>\### Proposition 10: Identify Gaps in Invoice Sequences</mark>**  

**<mark>\#### Problem:</mark>**  

**<mark>Find missing invoice numbers in sequences for better data integrity.</mark>**

In [22]:
USE WideWorldImporters;

WITH InvoiceSequence AS (
    SELECT 
        InvoiceID, 
        LAG(InvoiceID) OVER (ORDER BY InvoiceID) AS PreviousInvoice,
        InvoiceID - LAG(InvoiceID) OVER (ORDER BY InvoiceID) AS Gap
    FROM Sales.Invoices
)
SELECT * FROM InvoiceSequence
WHERE Gap > 1;


(0 rows affected)

Total execution time: 00:00:01.468

InvoiceID,PreviousInvoice,Gap
